In [5]:
!pip install "transformers<5.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [53]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

In [1]:
import transformers
print(transformers.__version__)

4.57.6


In [21]:
# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True, dtype=torch.float32)

In [22]:
sentences = ["How is the weather today?", "What is the current weather like today?"]
tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v3")
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
print(encoded_input)

{'input_ids': tensor([[    0, 11249,    83,    70, 92949, 18925,    32,     2,     1,     1],
        [    0,  4865,    83,    70, 43581, 92949,  1884, 18925,    32,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [23]:
with torch.no_grad():
    model_output = model(**encoded_input, adapter_mask=None)

print(model_output.last_hidden_state.shape)

torch.Size([2, 10, 1024])


In [24]:
print(model_output.last_hidden_state[:, :2, :3])

tensor([[[ 0.2714, -1.2857, -0.0474],
         [ 0.4978, -2.4512, -0.0046]],

        [[ 0.5941, -1.5652, -0.0543],
         [ 0.9864, -2.8542, -0.0122]]])


In [25]:
input_ids = encoded_input["input_ids"]

In [36]:
hub_model_embeddings = model.roberta.emb_ln(model.roberta.embeddings(input_ids))
# hub_model_embeddings = model.roberta.emb_ln(model.roberta.embeddings(input_ids))[:, :3, :10]

In [30]:
hub_model.dtype, my_model.dtype

(torch.float32, torch.float32)

In [32]:
my_model = torch.tensor([[[-0.0915, -0.1151, -0.3444, -0.2328,  0.4294,  0.0278,  0.1542,
           0.1918,  0.0476,  1.6633],
         [-0.3539, -0.1435,  1.1491,  0.4521,  0.0376, -0.5838,  0.8453,
          -0.3838, -0.0991,  0.1340],
         [ 0.1799,  0.2082,  0.6326, -0.4985, -0.2215,  0.0464,  0.8378,
          -0.5676, -0.2870,  0.6630]],

        [[-0.0915, -0.1151, -0.3444, -0.2328,  0.4294,  0.0278,  0.1542,
           0.1918,  0.0476,  1.6633],
         [-0.2000,  0.4030,  0.0077,  1.0833, -0.3007,  0.1863,  0.0059,
           0.1981, -0.3479,  0.9278],
         [ 0.1799,  0.2082,  0.6326, -0.4985, -0.2215,  0.0464,  0.8378,
          -0.5676, -0.2870,  0.6630]]])

is_match = torch.allclose(hub_model, my_model, atol=1e-4)

if is_match:
    print("✅ SUCCESS: The outputs match perfectly!")
else:
    print("❌ FAILURE: The outputs differ.")
    diff = (hub_model - my_model).abs().max().item()
    print(f"Max difference: {diff}")


✅ SUCCESS: The outputs match perfectly!


In [56]:
# This implementation was adapted from https://github.com/Dao-AILab/flash-attention/blob/main/flash_attn/modules/block.py
# Commit id: c94cd09744d20f0ac587a351ff6ff2e8ad11ae1b

# Previously adapted from https://github.com/mlcommons/training_results_v1.1/blob/main/NVIDIA/benchmarks/bert/implementations/pytorch/padding.py

import torch
import torch.nn.functional as F
from einops import rearrange, repeat


class IndexFirstAxis(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, indices):
        ctx.save_for_backward(indices)
        assert input.ndim >= 2
        ctx.first_axis_dim, other_shape = input.shape[0], input.shape[1:]
        second_dim = other_shape.numel()
        # TD [2022-03-04] For some reason torch.gather is a bit faster than indexing.
        # return input[indices]
        return torch.gather(
            rearrange(input, "b ... -> b (...)"),
            0,
            repeat(indices, "z -> z d", d=second_dim),
        ).reshape(-1, *other_shape)

    @staticmethod
    def backward(ctx, grad_output):
        (indices,) = ctx.saved_tensors
        assert grad_output.ndim >= 2
        other_shape = grad_output.shape[1:]
        grad_output = rearrange(grad_output, "b ... -> b (...)")
        grad_input = torch.zeros(
            [ctx.first_axis_dim, grad_output.shape[1]],
            device=grad_output.device,
            dtype=grad_output.dtype,
        )
        # TD [2022-03-04] For some reason torch.scatter is a bit faster than indexing.
        # grad_input[indices] = grad_output
        grad_input.scatter_(
            0, repeat(indices, "z -> z d", d=grad_output.shape[1]), grad_output
        )
        return grad_input.reshape(ctx.first_axis_dim, *other_shape), None


index_first_axis = IndexFirstAxis.apply


class IndexPutFirstAxis(torch.autograd.Function):
    @staticmethod
    def forward(ctx, values, indices, first_axis_dim):
        ctx.save_for_backward(indices)
        assert indices.ndim == 1
        assert values.ndim >= 2
        output = torch.zeros(
            first_axis_dim, *values.shape[1:], device=values.device, dtype=values.dtype
        )
        # TD [2022-03-04] For some reason torch.scatter is a bit faster than indexing.
        output[indices] = values
        # output.scatter_(0, repeat(indices, 'z -> z d', d=values.shape[1]), values)
        return output

    @staticmethod
    def backward(ctx, grad_output):
        (indices,) = ctx.saved_tensors
        # TD [2022-03-04] For some reason torch.gather is a bit faster than indexing.
        grad_values = grad_output[indices]
        # grad_values = torch.gather(grad_output, 0, repeat(indices, 'z -> z d', d=grad_output.shape[1]))
        return grad_values, None, None


index_put_first_axis = IndexPutFirstAxis.apply


class IndexFirstAxisResidual(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, indices):
        ctx.save_for_backward(indices)
        assert input.ndim >= 2
        ctx.first_axis_dim, other_shape = input.shape[0], input.shape[1:]
        second_dim = other_shape.numel()
        # TD [2022-03-04] For some reason torch.gather is a bit faster than indexing.
        output = input[indices]
        # We don't want to reshape input (b ... -> b (...)) since it could change the channel_last
        # memory format to channel_first. In other words, input might not be contiguous.
        # If we don't detach, Pytorch complains about output being a view and is being modified inplace
        return output, input.detach()

    @staticmethod
    def backward(ctx, grad_output, grad_residual):
        (indices,) = ctx.saved_tensors
        assert grad_output.ndim >= 2
        other_shape = grad_output.shape[1:]
        assert grad_residual.shape[1:] == other_shape
        grad_input = grad_residual
        # grad_input[indices] += grad_output
        indices = indices.reshape(indices.shape[0], *((1,) * (grad_output.ndim - 1)))
        indices = indices.expand_as(grad_output)
        grad_input.scatter_add_(0, indices, grad_output)
        return grad_input.reshape(ctx.first_axis_dim, *other_shape), None


index_first_axis_residual = IndexFirstAxisResidual.apply


def unpad_input(hidden_states, attention_mask, adapter_mask=None):
    """
    Arguments:
        hidden_states: (batch, seqlen, ...)
        attention_mask: (batch, seqlen), bool / int, 1 means valid and 0 means not valid.
    Return:
        hidden_states: (total_nnz, ...), where total_nnz = number of tokens in selected in attention_mask.
        indices: (total_nnz), the indices of non-masked tokens from the flattened input sequence.
        cu_seqlens: (batch + 1), the cumulative sequence lengths, used to index into hidden_states.
        max_seqlen_in_batch: int
    """
    seqlens_in_batch = attention_mask.sum(dim=-1, dtype=torch.int32)
    indices = torch.nonzero(attention_mask.flatten(), as_tuple=False).flatten()
    max_seqlen_in_batch = seqlens_in_batch.max().item()
    cu_seqlens = F.pad(
        torch.cumsum(seqlens_in_batch, dim=0, dtype=torch.torch.int32), (1, 0)
    )

    cu_adapter_mask = (
        torch.repeat_interleave(adapter_mask, cu_seqlens[1:] - cu_seqlens[:-1])
        if adapter_mask is not None
        else None
    )

    # TD [2022-03-04] We don't want to index with a bool mask, because Pytorch will expand the
    # bool mask, then call nonzero to get the indices, then index with those. The indices is @dim
    # times larger than it needs to be, wasting memory. It's faster and more memory-efficient to
    # index with integer indices. Moreover, torch's index is a bit slower than it needs to be,
    # so we write custom forward and backward to make it a bit faster.
    return (
        index_first_axis(rearrange(hidden_states, "b s ... -> (b s) ..."), indices),
        indices,
        cu_seqlens,
        max_seqlen_in_batch,
        cu_adapter_mask,
    )


def unpad_input_for_concatenated_sequences(hidden_states, attention_mask_in_length):
    """
    Supports concatenating short samples in one sequence. The attention_mask_in_length is utilized to mask other short samples. It helps efficient training of variant lengths-based samples (e.g., the supervised fine-tuning task in large language model).
    The motivation for this function is explained [here](https://github.com/Dao-AILab/flash-attention/issues/432#issuecomment-1668822286).
    For example, if batch = 3 and seqlen = 6, the attention_mask_in_length is:
        ```
        [
          [2, 3, 0, 0, 0, 0],
          [3, 2, 0, 0, 0, 0],
          [6, 0, 0, 0, 0, 0]
        ]
        ```
    , which refers to the 3D-attention mask:
        ```
        [
          [
            [1, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0],
            [0, 0, 1, 1, 0, 0],
            [0, 0, 1, 1, 1, 0],
            [0, 0, 0, 0, 0, 1]
          ],
          [
            [1, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0],
            [0, 0, 0, 1, 0, 0],
            [0, 0, 0, 1, 1, 0],
            [0, 0, 0, 0, 0, 1]
          ],
          [
            [1, 0, 0, 0, 0, 0],
            [1, 1, 0, 0, 0, 0],
            [1, 1, 1, 0, 0, 0],
            [1, 1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1, 0],
            [1, 1, 1, 1, 1, 1]
          ]
        ]
        ```.
    Arguments:
        hidden_states: (batch, seqlen, ...)
        attention_mask_in_length: (batch, seqlen), int, a nonzero number (e.g., 1, 2, 3, etc.) means length of concatenated sequence in b-th batch, and 0 means none.
    Return:
        hidden_states: (total_nnz, ...), where total_nnz = number of tokens in selected in attention_mask.
        indices: (total_nnz), the indices of non-masked tokens from the flattened input sequence.
        cu_seqlens: (batch + 1), the cumulative sequence lengths, used to index into hidden_states.
        max_seqlen_in_batch: int
    """
    length = attention_mask_in_length.sum(dim=-1)
    seqlen = attention_mask_in_length.size(-1)
    attention_mask_2d = torch.arange(
        seqlen, device=length.device, dtype=length.dtype
    ).expand(len(length), seqlen) < length.unsqueeze(1)
    real_indices_idx = torch.nonzero(
        attention_mask_in_length.flatten(), as_tuple=False
    ).flatten()
    seqlens_in_batch = attention_mask_in_length.flatten()[real_indices_idx]
    indices = torch.nonzero(attention_mask_2d.flatten(), as_tuple=False).flatten()
    max_seqlen_in_batch = seqlens_in_batch.max().item()
    cu_seqlens = F.pad(
        torch.cumsum(seqlens_in_batch, dim=0, dtype=torch.torch.int32), (1, 0)
    )
    # TD [2022-03-04] We don't want to index with a bool mask, because Pytorch will expand the
    # bool mask, then call nonzero to get the indices, then index with those. The indices is @dim
    # times larger than it needs to be, wasting memory. It's faster and more memory-efficient to
    # index with integer indices. Moreover, torch's index is a bit slower than it needs to be,
    # so we write custom forward and backward to make it a bit faster.
    return (
        index_first_axis(rearrange(hidden_states, "b s ... -> (b s) ..."), indices),
        indices,
        cu_seqlens,
        max_seqlen_in_batch,
    )


def pad_input(hidden_states, indices, batch, seqlen):
    """
    Arguments:
        hidden_states: (total_nnz, ...), where total_nnz = number of tokens in selected in attention_mask.
        indices: (total_nnz), the indices that represent the non-masked tokens of the original padded input sequence.
        batch: int, batch size for the padded sequence.
        seqlen: int, maximum sequence length for the padded sequence.
    Return:
        hidden_states: (batch, seqlen, ...)
    """
    dim = hidden_states.shape[-1]
    # output = torch.zeros((batch * seqlen), dim, device=hidden_states.device, dtype=hidden_states.dtype)
    # output[indices] = hidden_states
    output = index_put_first_axis(hidden_states, indices, batch * seqlen)
    return rearrange(output, "(b s) ... -> b s ...", b=batch)


In [66]:
hub_layer0 = model.roberta.encoder.layers[0]

# hidden_states, indices, cu_seqlens, max_seqlen_in_batch, cu_adapter_mask = unpad_input(
#     hub_model_embeddings, attention_mask=encoded_input["attention_mask"], adapter_mask=None
# )


# mixer_kwargs = {
#     "cu_seqlens": cu_seqlens,
#     "max_seqlen": max_seqlen_in_batch,
#     "adapter_mask": cu_adapter_mask,
# }

mixer_kwargs = {"adapter_mask": None}


with torch.no_grad():
    hub_layer_out = hub_layer0(
        hub_model_embeddings,
        mixer_kwargs=mixer_kwargs,
    )

print(hub_layer_out)

tensor([[[-0.1008, -0.0022, -0.0984,  ..., -0.0865, -0.1413, -0.1898],
         [-0.4567,  0.3215,  0.9694,  ...,  0.2264,  0.7275, -0.3495],
         [ 0.1945, -0.1620,  0.3424,  ..., -0.5715,  0.4735, -0.1320],
         ...,
         [-0.1147,  0.0201, -0.0754,  ..., -0.0488, -0.0517, -0.0738],
         [ 0.1777, -0.0335,  0.2197,  ..., -0.1171, -0.2434,  0.1163],
         [ 0.1794, -0.0350,  0.2183,  ..., -0.1137, -0.2428,  0.1188]],

        [[-0.0938, -0.0058, -0.0988,  ..., -0.0919, -0.1419, -0.1836],
         [-0.1884,  0.4849,  0.0168,  ...,  0.2507,  0.8042,  0.3683],
         [ 0.0998,  0.0811, -0.0847,  ..., -0.0879,  0.8617, -0.0685],
         ...,
         [ 0.3443,  0.6123,  0.6049,  ...,  0.2390,  0.2462,  0.0592],
         [ 0.0834, -0.0496,  0.3199,  ..., -0.0139,  0.2958, -0.0910],
         [-0.1137,  0.0203, -0.0787,  ..., -0.0516, -0.0510, -0.0685]]])


In [63]:
hub_layer0

Block(
  (mixer): MHA(
    (rotary_emb): RotaryEmbedding()
    (Wqkv): ParametrizedLinearResidual(
      in_features=1024, out_features=3072, bias=True
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): LoRAParametrization()
        )
      )
    )
    (inner_attn): SelfAttention(
      (drop): Dropout(p=0.1, inplace=False)
    )
    (inner_cross_attn): CrossAttention(
      (drop): Dropout(p=0.1, inplace=False)
    )
    (out_proj): ParametrizedLinear(
      in_features=1024, out_features=1024, bias=True
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): LoRAParametrization()
        )
      )
    )
  )
  (dropout1): Dropout(p=0.1, inplace=False)
  (drop_path1): StochasticDepth(p=0.0, mode=row)
  (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (mlp): Mlp(
    (fc1): ParametrizedLinear(
      in_features=1024, out_features=4096, bias=True
      (parametrizations): ModuleDict(
        (we

In [64]:
# l0_out[:, :3, :10]
my_model = torch.tensor([[[-0.0920, -0.0060, -0.0988, -0.0747,  0.1262, -0.0315, -0.0270,
           0.0741,  0.3227,  0.1976],
         [-0.3754,  0.3618,  1.0853,  0.2226,  0.1287, -0.4894,  0.6106,
          -0.5463, -0.1336, -0.1300],
         [ 0.1316,  0.0256, -0.0398, -0.3055, -0.2006,  0.0590,  0.7749,
          -0.1645, -0.0344,  0.2116]],

        [[-0.0938, -0.0058, -0.0988, -0.0741,  0.1144, -0.0333, -0.0269,
           0.0655,  0.3169,  0.1977],
         [-0.1884,  0.4849,  0.0168,  1.1222, -0.3220, -0.0470, -0.1166,
           0.5366, -0.3542,  0.4593],
         [ 0.0998,  0.0811, -0.0847, -0.3566, -0.2404,  0.0257,  0.8107,
          -0.1530, -0.1304,  0.2161]]])

hub_layer_out = hub_layer_out[:, :3, :10]


is_match = torch.allclose(hub_layer_out, my_model, atol=1e-4)

if is_match:
    print("✅ SUCCESS: The outputs match perfectly!")
else:
    print("❌ FAILURE: The outputs differ.")
    diff = (hub_model - my_model).abs().max().item()
    print(f"Max difference: {diff}")

❌ FAILURE: The outputs differ.
Max difference: 1.4656792879104614
